# Day 39: Simulate Tensor Parallelism: Split MatMul Across N Workers

**Layer:** Implementation | **Prerequisite:** Previous days


## Concept Overview

Implements column-parallel and row-parallel matrix multiplication, simulating how tensor parallelism splits a single matmul across N GPUs. Verifies correctness against single-GPU reference and models AllReduce communication overhead.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Tensor Parallelism: Split MatMul Across N Workers


In [ ]:
def column_parallel_matmul(X, W, n_workers):
    """
    Column-parallel: split W by output dimension.
    X: [batch, d_in]
    W: [d_out, d_in]
    Returns: [batch, d_out] via all-gather
    """
    d_out = W.shape[0]
    assert d_out % n_workers == 0
    chunk = d_out // n_workers

    # Each worker computes its chunk
    partial_outputs = []
    for i in range(n_workers):
        W_i = W[i*chunk:(i+1)*chunk, :]
        partial_outputs.append(X @ W_i.T)  # [batch, chunk]

    # All-gather: concatenate outputs
    return torch.cat(partial_outputs, dim=-1)

def row_parallel_matmul(X, W, n_workers):
    """
    Row-parallel: split W by input dimension.
    X: [batch, d_in], W: [d_out, d_in]
    Returns: [batch, d_out] via all-reduce
    """
    d_in = W.shape[1]
    assert d_in % n_workers == 0
    chunk = d_in // n_workers

    # Each worker has its slice of X and corresponding rows of W
    partial_outputs = []
    for i in range(n_workers):
        X_i = X[:, i*chunk:(i+1)*chunk]
        W_i = W[:, i*chunk:(i+1)*chunk]
        partial_outputs.append(X_i @ W_i.T)  # [batch, d_out]

    # All-reduce: sum partial outputs
    return sum(partial_outputs)

# Verify both produce correct results
torch.manual_seed(42)
batch, d_in, d_out = 16, 512, 1024
X = torch.randn(batch, d_in)
W = torch.randn(d_out, d_in)
ref = X @ W.T

for n in [1, 2, 4, 8]:
    out_col = column_parallel_matmul(X, W, n)
    out_row = row_parallel_matmul(X, W, n)
    col_ok = torch.allclose(ref, out_col, atol=1e-4)
    row_ok = torch.allclose(ref, out_row, atol=1e-4)
    print(f'N={n}: column_parallel={col_ok}, row_parallel={row_ok}')


## 2. Communication Overhead Modeling


In [ ]:
import time

def simulate_tp_matmul(d_model, d_ff, batch, n_workers, bandwidth_gb_s=900):
    """Model time for tensor-parallel FFN with communication."""
    # Compute time: scaled by 1/n_workers
    flops_per_worker = 2 * batch * d_model * d_ff / n_workers
    compute_s = flops_per_worker / 67e12  # DGX Spark GB10 ~67 TFLOPS FP16

    # Communication: AllReduce after each row-parallel layer
    # AllReduce volume = batch * d_model * dtype_bytes
    allreduce_bytes = batch * d_model * 2  # FP16
    allreduce_s = 2 * (n_workers-1)/n_workers * allreduce_bytes / (bandwidth_gb_s * 1e9)

    return compute_s * 1000, allreduce_s * 1000  # ms

print('Tensor Parallel FFN time modeling (Llama-3-8B layer):')
print(f'{"Workers":>8} {"Compute (ms)":>14} {"AllReduce (ms)":>16} {"Overhead %":>12}')
for n in [1, 2, 4, 8]:
    comp, comm = simulate_tp_matmul(4096, 14336, 1, n)
    overhead = comm / (comp + comm) * 100
    print(f'{n:>8} {comp:>14.3f} {comm:>16.4f} {overhead:>11.1f}%')


## Experiments

1. Measure actual matmul speedup on your hardware. Is it linear with GPU count?
2. Implement Megatron-style paired column+row parallel to reduce AllReduce from 2 to 1 per attention block.
3. Profile the actual NVLink bandwidth between spark-01 and spark-02 using torch.distributed.


## Key Takeaways

- See concept overview above for the key points.
- Day 39 complete.
